In [44]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

import joblib

from xgboost import XGBClassifier
from catboost import CatBoostClassifier


In [2]:
# Load the previously trained Random Forest pipeline.
# This restores the preprocessing and model without retraining.

best_rf = joblib.load("best_random_forest.joblib")

print("Best Random Forest loaded successfully.")

Best Random Forest loaded successfully.


In [3]:
# importing datasets
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

In [4]:
train.shape, test.shape

((691369, 14), (296302, 13))

### **1. Understanding the Dataset**

Before building any model, we first inspect the structure of the dataset and identify:

- The target variable (`addicted_label`)
- Numerical and categorical features
- Missing values
- The distribution of the target

This first step helps us understand what kind of prediction problem we are dealing with and prevents us from making modeling decisions before understanding the data.

In [5]:
train.head()

,id,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,gender,stress_level,academic_work_impact,addicted_label
0,0,24.0,NaN,1.83,1.59,2.11,7.46,122.0,38.0,8.63,Male,Medium,No,1
1,1,19.0,5.97,1.08,NaN,3.03,8.22,76.0,19.0,NaN,Female,Medium,No,0
2,2,18.0,5.09,NaN,NaN,NaN,6.25,134.0,60.0,7.47,Female,Low,Yes,0
3,3,21.0,6.42,1.26,1.42,3.36,8.85,112.0,94.0,8.66,Other,Low,NaN,1
4,4,26.0,11.20,1.87,2.81,1.95,5.25,NaN,NaN,13.39,Female,Medium,No,1


In [6]:
# see if we have any missing values
# also checking for the right column type
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       691369 non-null  int64  
 1   age                      662440 non-null  float64
 2   daily_screen_time_hours  595515 non-null  float64
 3   social_media_hours       557374 non-null  float64
 4   gaming_hours             564548 non-null  float64
 5   work_study_hours         639851 non-null  float64
 6   sleep_hours              646889 non-null  float64
 7   notifications_per_day    623785 non-null  float64
 8   app_opens_per_day        610659 non-null  float64
 9   weekend_screen_time      579306 non-null  float64
 10  gender                   662335 non-null  str    
 11  stress_level             636221 non-null  str    
 12  academic_work_impact     647145 non-null  str    
 13  addicted_label           691369 non-null  int64  
dtypes: float64(9), 

### **Target Distribution**

In [7]:
# checking how unbalanced our dataset is
train["addicted_label"].value_counts(normalize=True)

addicted_label
1    0.709424
0    0.290576
Name: proportion, dtype: float64

The target variable is moderately imbalanced:

- `1` (addicted): **70.94%**
- `0` (not addicted): **29.06%**

This means that the positive class is considerably more frequent than the negative class.

However, our competition metric is **ROC AUC**, which evaluates how well the model ranks positive observations above negative observations. Therefore, the class imbalance is less concerning than it would be for a metric such as raw accuracy.

At this stage, we will keep the original class distribution and investigate whether any special handling is necessary later.

### **Missing Values**

In [8]:
# back to missing values
# let's see if we can find a pattern among different columns
train.isna().mean().sort_values(ascending=False)

social_media_hours         0.193811
gaming_hours               0.183435
weekend_screen_time        0.162089
daily_screen_time_hours    0.138644
app_opens_per_day          0.116739
notifications_per_day      0.097754
stress_level               0.079766
work_study_hours           0.074516
sleep_hours                0.064336
academic_work_impact       0.063966
gender                     0.041995
age                        0.041843
id                         0.000000
addicted_label             0.000000
dtype: float64

In [9]:
# checking whether the missing values are linked to one category
train.groupby("addicted_label").apply(
    lambda x: x.isna().mean()
)

,id,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,gender,stress_level,academic_work_impact
addicted_label,,,,,,,,,,,,,
0,0.0,0.041270,0.137749,0.193818,0.182752,0.074203,0.063461,0.097479,0.115389,0.161403,0.042087,0.079883,0.063959
1,0.0,0.042078,0.139010,0.193808,0.183714,0.074644,0.064695,0.097867,0.117293,0.162369,0.041957,0.079719,0.063969


In [10]:
top_features = [
    "daily_screen_time_hours",
    "weekend_screen_time",
    "stress_level",
    "social_media_hours",
    "academic_work_impact"
]

In [11]:
train.groupby("addicted_label")[
    ["daily_screen_time_hours",
     "weekend_screen_time",
     "social_media_hours"]
].mean()

,daily_screen_time_hours,weekend_screen_time,social_media_hours
addicted_label,,,
0,5.042929,6.848920,1.376144
1,8.706523,10.558729,2.919495


Several numerical features contain a substantial proportion of missing values.

The highest missing-value rates are found in:

- `social_media_hours`
- `gaming_hours`
- `weekend_screen_time`
- `daily_screen_time_hours`

We compared missing-value rates between the two target classes and found almost no meaningful difference.

This suggests that missingness itself does not appear to be strongly associated with `addicted_label`.

Therefore, we should **not automatically treat missingness as a predictive signal**, but we also should not discard the rows. We will decide how to handle missing values based on the models we test later.

### **Categorical Features vs. Target**

In [12]:
pd.crosstab(
    train["stress_level"],
    train["addicted_label"],
    normalize="index"
)

addicted_label,0,1
stress_level,,
High,0.288569,0.711431
Low,0.288840,0.711160
Medium,0.294337,0.705663


In [13]:
pd.crosstab(
    train["academic_work_impact"],
    train["addicted_label"],
    normalize="index"
)

addicted_label,0,1
academic_work_impact,,
No,0.288971,0.711029
Yes,0.292117,0.707883


We investigated `stress_level` and `academic_work_impact`, as I supposed they'll be linked to the target variable.

Instead, we found that their target distributions were extremely similar across categories:

- `stress_level`: approximately **71% addicted** for every category
- `academic_work_impact`: approximately **71% addicted** for both categories

This means that these variables show very little predictive signal **on their own**.

Although they were intuitively plausible predictors, the data does not currently support that hypothesis.

We will keep them for now rather than removing them prematurely, because machine learning models can sometimes extract information from interactions that is not visible from marginal statistics.

### **Relationships Between Numerical Features**

In [14]:
# We want to measure how strongly our main numerical features are related to each other
#
# If two variables have a very high correlation, they may be providing similar information to the model

features_numeric = [
    "daily_screen_time_hours",
    "weekend_screen_time",
    "social_media_hours",
    "gaming_hours",
    "notifications_per_day",
    "app_opens_per_day",
    "work_study_hours",
    "sleep_hours",
    "age"
]

# Compute the correlation matrix between numerical features
correlation_matrix = train[features_numeric].corr()

# Round the results to make the matrix easier to read
correlation_matrix.round(2)

,daily_screen_time_hours,weekend_screen_time,social_media_hours,gaming_hours,notifications_per_day,app_opens_per_day,work_study_hours,sleep_hours,age
daily_screen_time_hours,1.00,0.80,0.60,0.43,0.00,0.05,0.53,0.03,0.01
weekend_screen_time,0.80,1.00,0.48,0.34,-0.00,0.05,0.42,0.03,0.01
social_media_hours,0.60,0.48,1.00,0.09,-0.01,0.05,0.10,0.02,0.01
gaming_hours,0.43,0.34,0.09,1.00,0.02,0.02,0.05,0.02,0.00
notifications_per_day,0.00,-0.00,-0.01,0.02,1.00,0.01,-0.01,0.03,-0.01
app_opens_per_day,0.05,0.05,0.05,0.02,0.01,1.00,0.01,0.02,0.00
work_study_hours,0.53,0.42,0.10,0.05,-0.01,0.01,1.00,-0.00,0.00
sleep_hours,0.03,0.03,0.02,0.02,0.03,0.02,-0.00,1.00,0.00
age,0.01,0.01,0.01,0.00,-0.01,0.00,0.00,0.00,1.00


We calculated the correlation matrix to understand how numerical features relate to each other.

Several interesting relationships emerged:

- `daily_screen_time_hours` and `weekend_screen_time`: **0.80**
- `daily_screen_time_hours` and `social_media_hours`: **0.60**
- `daily_screen_time_hours` and `work_study_hours`: **0.53**
- `daily_screen_time_hours` and `gaming_hours`: **0.43**

The strong correlation between daily and weekend screen time suggests that these features capture related aspects of overall phone usage.

However, high correlation between features does **not** automatically mean that one should be removed. A feature can still provide additional predictive information when combined with another feature.

Interestingly, `notifications_per_day`, `app_opens_per_day`, `sleep_hours`, and `age` show very little correlation with the other numerical features. This does not prove that they are useless, because nonlinear relationships or interactions may still make them valuable to a model.

### **Numerical Features vs. Target**

In [15]:
# We want to measure how strongly each numerical feature is linearly associated with our target.
#
# Just first, quick way to identify variables that may contain predictive signal.

target_correlation = (
    train[features_numeric + ["addicted_label"]]
    .corr()["addicted_label"]
    .drop("addicted_label")
    .sort_values(key=abs, ascending=False)
)

# Display the strongest correlations first.
target_correlation.round(3)

daily_screen_time_hours    0.611
weekend_screen_time        0.590
social_media_hours         0.532
work_study_hours           0.251
gaming_hours               0.205
app_opens_per_day          0.063
sleep_hours                0.043
notifications_per_day     -0.012
age                        0.004
Name: addicted_label, dtype: float64

We next measured the linear correlation between each numerical feature and `addicted_label` as a first indication of predictive signal.

The strongest relationships were:

1. `daily_screen_time_hours` → **0.611**
2. `weekend_screen_time` → **0.590**
3. `social_media_hours` → **0.532**
4. `work_study_hours` → **0.251**
5. `gaming_hours` → **0.205**

The remaining variables showed very weak linear correlations with the target.

These results strongly support our initial hypothesis that overall screen usage and specific types of phone usage contain substantial predictive information.

However, correlation alone is not feature importance. A variable with weak individual correlation may still improve model performance through nonlinear effects or interactions with other features.

### **Daily Screen Time and Addiction**

In [16]:
# We will divide daily screen time into quantile-based bins.
# Each bin contains roughly the same number of observations.
#
# Then we calculate the average target within each bin.
# This lets us see how the probability of addiction changes
# as screen time increases.

screen_time_bins = pd.qcut(
    train["daily_screen_time_hours"],
    q=10,
    duplicates="drop"
)

screen_time_target = (
    train.groupby(screen_time_bins, observed=False)["addicted_label"]
    .mean()
)

# Display the average target rate for each screen-time range.
screen_time_target

daily_screen_time_hours
(0.499, 3.97]     0.265061
(3.97, 4.96]      0.310385
(4.96, 5.89]      0.398567
(5.89, 6.8]       0.534341
(6.8, 7.77]       0.723417
(7.77, 8.7]       0.895326
(8.7, 9.47]       0.974265
(9.47, 10.25]     0.994738
(10.25, 11.24]    0.998661
(11.24, 15.0]     0.999661
Name: addicted_label, dtype: float64

We divided `daily_screen_time_hours` into quantile-based groups and calculated the proportion of addicted users in each group.

The addiction rate increases sharply as screen time increases:

- Around 4 hours → ~27%
- Around 6 hours → ~53%
- Around 7.8 hours → ~72%
- Around 8.7 hours → ~90%
- Above 9.5 hours → ~97%+
- Above 11 hours → ~100%

This reveals a strong **monotonic but nonlinear relationship**.

The pattern resembles a sigmoid curve: the probability of addiction increases rapidly in the middle range and eventually approaches 100%.

This suggests that a purely linear model may not capture the relationship efficiently, while models capable of learning nonlinear patterns may have an advantage.

### **Weekend Screen Time and Addiction**

In [17]:
# Divide weekend screen time into 10 quantile-based groups.
# Each group contains roughly the same number of observations.
#
# We then calculate the addiction rate in each group to see
# whether weekend usage has a similar predictive pattern
# to daily screen time.

weekend_bins = pd.qcut(
    train["weekend_screen_time"],
    q=10,
    duplicates="drop"
)

weekend_target = (
    train.groupby(weekend_bins, observed=False)["addicted_label"]
    .mean()
)

# Show the average addiction rate for each weekend screen-time range.
weekend_target

weekend_screen_time
(0.509, 5.63]     0.264420
(5.63, 6.76]      0.322108
(6.76, 7.73]      0.396198
(7.73, 8.68]      0.572781
(8.68, 9.58]      0.746412
(9.58, 10.47]     0.867965
(10.47, 11.33]    0.948534
(11.33, 12.08]    0.988054
(12.08, 13.19]    0.997962
(13.19, 17.56]    0.999689
Name: addicted_label, dtype: float64

We repeated the same analysis for `weekend_screen_time`.

The resulting pattern is very similar to daily screen time:

- Around 5.6 hours → ~26%
- Around 7.7 hours → ~40%
- Around 8.7 hours → ~57%
- Around 9.6 hours → ~75%
- Around 10.5 hours → ~87%
- Above 12 hours → ~99%+

Again, the relationship is strongly monotonic and nonlinear.

The saturation point occurs at a somewhat higher value than for daily screen time, but the overall shape is remarkably similar.

This suggests that both variables capture important information about phone usage, while their different scales and thresholds indicate that they may not be completely interchangeable.

### **Numerical Feature Distributions**

In [18]:
# We want a quick summary of the numerical features:
# minimum, maximum, quartiles, mean, and standard deviation.
#
# This helps us detect:
# - unusual ranges
# - potential outliers
# - variables with very different scales
# - suspicious values that may require further investigation

train[features_numeric].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
daily_screen_time_hours,595515.0,7.64,2.72,0.50,5.48,7.77,9.84,15.00
weekend_screen_time,579306.0,9.48,2.86,0.51,7.28,9.58,11.75,17.56
social_media_hours,557374.0,2.47,1.32,0.00,1.45,2.31,3.37,8.00
gaming_hours,564548.0,1.46,0.93,0.00,0.70,1.33,2.09,4.00
notifications_per_day,623785.0,145.89,65.92,20.00,93.00,150.00,204.00,250.00
app_opens_per_day,610659.0,102.64,48.09,15.00,64.00,104.00,145.00,180.00
work_study_hours,639851.0,2.37,1.26,0.00,1.36,2.20,3.20,6.00
sleep_hours,646889.0,6.80,1.23,4.50,5.78,6.80,7.87,9.00
age,662440.0,26.62,5.15,18.00,22.00,27.00,31.00,35.00


We inspected the descriptive statistics of the numerical features to identify unusual ranges, strong skewness, and potential outliers.

The features appear to have plausible and relatively constrained ranges. For example:

- `age` ranges from 18 to 35
- `daily_screen_time_hours` ranges from 0.5 to 15 hours
- `weekend_screen_time` ranges from 0.5 to 17.56 hours
- `social_media_hours` ranges from 0 to 8 hours
- `gaming_hours` ranges from 0 to 4 hours

For most variables, the mean and median are relatively close, suggesting that there are no obvious signs of severe skewness.

However, mean and median alone are not sufficient to rule out statistical outliers. We will therefore avoid aggressive outlier removal unless a later model or visualization provides evidence that it is necessary.

The numerical features also operate on different scales. This will be relevant for scale-sensitive models, but it is less concerning for tree-based models, which we expect to test later.

### **Train-Validation Split**

In [19]:
# Separate the features from the target.
X = train.drop(columns="addicted_label")
y = train["addicted_label"]

# Split the data into training and validation sets.
# We use stratify=y so both sets keep approximately the same
# proportion of addicted and non-addicted observations.
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Check the resulting shapes.
print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)

# Confirm that the target distribution stayed consistent.
print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nValidation target distribution:")
print(y_valid.value_counts(normalize=True))

X_train: (553095, 13)
X_valid: (138274, 13)

Training target distribution:
addicted_label
1    0.709424
0    0.290576
Name: proportion, dtype: float64

Validation target distribution:
addicted_label
1    0.709425
0    0.290575
Name: proportion, dtype: float64


Before training any model, we split the training dataset into two parts:

- **80% training data**: used by the model to learn patterns.
- **20% validation data**: kept unseen during training and used to estimate how well the model generalizes to new data.

We use a **stratified split** so that both subsets preserve approximately the same proportion of positive and negative target values.

This is important because our target is moderately imbalanced:

- ~71% `addicted_label = 1`
- ~29% `addicted_label = 0`

We also set `random_state=42` so that our experiments are reproducible and different models can be compared using the exact same split.

### **Baseline Model: Logistic Regression**

In [20]:
# Identify numerical and categorical columns.
# We exclude "id" because it is an identifier, not a meaningful feature.
numeric_features = [
    "age",
    "daily_screen_time_hours",
    "social_media_hours",
    "gaming_hours",
    "work_study_hours",
    "sleep_hours",
    "notifications_per_day",
    "app_opens_per_day",
    "weekend_screen_time"
]

categorical_features = [
    "gender",
    "stress_level",
    "academic_work_impact"
]

# Numerical preprocessing:
# Missing numerical values are replaced with the median.
# The median is robust to potential skewness and is simple for our baseline.
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

# Categorical preprocessing:
# Missing categories are replaced with the most frequent category,
# then categorical values are converted into binary indicator columns.
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

# Combine both preprocessing strategies.
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# Build the complete pipeline:
# preprocessing happens first, then Logistic Regression is trained.
baseline_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000))
    ]
)


In [21]:
# Train the baseline model using only the training split.
baseline_model.fit(X_train, y_train)

# Generate probabilities for the positive class on validation data.
# ROC AUC needs continuous scores rather than hard 0/1 predictions.
valid_predictions = baseline_model.predict_proba(X_valid)[:, 1]

# Evaluate the baseline
baseline_auc = roc_auc_score(y_valid, valid_predictions)

print(f"Baseline ROC AUC: {baseline_auc:.5f}")

Baseline ROC AUC: 0.91028


We start with Logistic Regression as a simple and interpretable baseline.

The model receives:

- Numerical features with missing values imputed using the median.
- Categorical features with missing values imputed using the most frequent category.
- Categorical variables converted into numerical indicators using One-Hot Encoding.

All preprocessing is placed inside a single `Pipeline` so that the transformations are learned only from the training data, preventing data leakage.

We evaluate the model using **ROC AUC**.

This baseline is not expected to be our strongest model. Its purpose is to establish a reliable reference point.

From this point forward, every modeling experiment should be compared against this baseline using the same validation split and the same ROC AUC metric.

### **Feature Contribution: Top 3 Features**

Our first Logistic Regression achieved a ROC AUC of **0.91028** using all available features.

We now train a second Logistic Regression using only:

- `daily_screen_time_hours`
- `weekend_screen_time`
- `social_media_hours`

These features had the strongest individual relationships with the target during EDA.

The purpose of this experiment is not to create a final model, but to estimate how much predictive information is already contained in these three variables.

In [22]:
# We will train a simpler Logistic Regression using only the
# three features that showed the strongest individual signal.
#
# This experiment helps us answer:
# "How much of the predictive power comes from these three
# main screen-usage variables alone?"

top_features = [
    "daily_screen_time_hours",
    "weekend_screen_time",
    "social_media_hours"
]

# Select only the three strongest numerical features.
X_top = train[top_features]

# Use the same target as before.
y = train["addicted_label"]

# Split the reduced dataset using exactly the same strategy
# as our original baseline.
X_train_top, X_valid_top, y_train_top, y_valid_top = train_test_split(
    X_top,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Create a simple preprocessing pipeline for numerical features.
# We only need median imputation because all three features are numerical.
top_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

# Build the reduced Logistic Regression pipeline.
top_baseline_model = Pipeline(
    steps=[
        ("preprocessor", top_preprocessor),
        ("model", LogisticRegression(max_iter=1000))
    ]
)

# Train the model using only the three main features.
top_baseline_model.fit(X_train_top, y_train_top)

# Generate positive-class probabilities for validation data.
top_valid_predictions = top_baseline_model.predict_proba(
    X_valid_top
)[:, 1]

# Evaluate using the competition metric.
top_baseline_auc = roc_auc_score(
    y_valid_top,
    top_valid_predictions
)

print(f"Top-3 ROC AUC: {top_baseline_auc:.5f}")
print(f"Full baseline ROC AUC: {baseline_auc:.5f}")
print(f"Difference: {top_baseline_auc - baseline_auc:+.5f}")

Top-3 ROC AUC: 0.90984
Full baseline ROC AUC: 0.91028
Difference: -0.00044


Using only the three strongest features, we achieved a ROC AUC of **0.90984**.

The difference is only **0.00044**, which indicates that these three variables contain almost all of the predictive signal captured by our current baseline.

This is an important finding from the EDA:

> Most of the predictive information in this dataset appears to come from different measures of screen usage.

However, the Logistic Regression model is limited in its ability to capture nonlinear relationships. Our previous analysis showed that both daily and weekend screen time have a strong nonlinear relationships with addiction.

Therefore, our next experiment will keep the same features and validation split but replace Logistic Regression with a **tree-based model capable of learning nonlinear relationships**.

This isolates the effect of model complexity and allows us to determine whether the nonlinear patterns discovered during EDA can improve our ROC AUC.

### **First Nonlinear Model: Decision Tree**

We now replace Logistic Regression with a Decision Tree while keeping:

- the same three features
- the same train-validation split
- the same preprocessing
- the same evaluation metric

This isolates the effect of allowing the model to learn nonlinear relationships.

A Decision Tree can naturally learn threshold-based rules such as:

- screen time above a certain value
- weekend usage above another threshold
- combinations of different conditions

This makes it a useful first test of our hypothesis:

> If the nonlinear patterns observed during EDA contain additional predictive information, a tree-based model should outperform Logistic Regression.

We will compare the Decision Tree ROC AUC directly against our previous baseline.

In [23]:
# Create a small Decision Tree as our first nonlinear model.
# We start with a limited depth so the model remains simple
# and easier to interpret.
tree_model = Pipeline(
    steps=[
        ("preprocessor", top_preprocessor),
        ("model", DecisionTreeClassifier(
            max_depth=5,
            random_state=42
        ))
    ]
)

# Train the tree using the same Top-3 features and the same
# training/validation split as our Logistic Regression.
tree_model.fit(X_train_top, y_train_top)

# Generate probability scores for the positive class.
tree_valid_predictions = tree_model.predict_proba(
    X_valid_top
)[:, 1]

# Evaluate using the same ROC AUC metric.
tree_auc = roc_auc_score(
    y_valid_top,
    tree_valid_predictions
)

print(f"Decision Tree ROC AUC: {tree_auc:.5f}")
print(f"Logistic Regression ROC AUC: {top_baseline_auc:.5f}")
print(f"Improvement: {tree_auc - top_baseline_auc:+.5f}")

Decision Tree ROC AUC: 0.91864
Logistic Regression ROC AUC: 0.90984
Improvement: +0.00880


### **Nonlinear Model vs. Logistic Regression**

The Decision Tree achieved a ROC AUC of **0.91864**, compared with **0.90984** for Logistic Regression using the same three features.

This represents an improvement of **0.00880 ROC AUC**.

The result supports our earlier hypothesis that the relationship between screen usage and addiction is strongly nonlinear.

A Decision Tree can learn threshold-based patterns, such as different addiction probabilities above or below certain screen-time levels. Logistic Regression is much more limited because it models the relationship using a linear decision function.

This experiment demonstrates that the nonlinear structure identified during EDA contains real predictive value.

However, a single shallow Decision Tree is still a relatively simple model. Our next goal is to determine whether a more flexible tree-based ensemble can capture additional patterns without overfitting.

### **Decision Tree Complexity**

We now test several values of `max_depth` to understand how tree complexity affects validation performance.

We therefore expect validation ROC AUC to improve initially, reach a useful range, and potentially decrease if the tree becomes too complex.

The goal is not simply to maximize model complexity, but to find the complexity that generalizes best to unseen data.

In [24]:
# We will test several tree depths to study the trade-off
# between model complexity and generalization.
#
# Smaller depths may underfit because the tree cannot learn
# enough patterns.
#
# Larger depths are more flexible, but they may start fitting
# noise instead of general patterns.

tree_depths = [3, 5, 7, 10]

for depth in tree_depths:

    # Build a Decision Tree with the current depth.
    tree = Pipeline(
        steps=[
            ("preprocessor", top_preprocessor),
            ("model", DecisionTreeClassifier(
                max_depth=depth,
                random_state=42
            ))
        ]
    )

    # Train the model on the same training split.
    tree.fit(X_train_top, y_train_top)

    # Predict probabilities for the validation set.
    predictions = tree.predict_proba(X_valid_top)[:, 1]

    # Evaluate using ROC AUC.
    auc = roc_auc_score(y_valid_top, predictions)

    print(f"max_depth={depth:2d} | Validation ROC AUC={auc:.5f}")

max_depth= 3 | Validation ROC AUC=0.89531
max_depth= 5 | Validation ROC AUC=0.91864
max_depth= 7 | Validation ROC AUC=0.92792
max_depth=10 | Validation ROC AUC=0.92967


### **Decision Tree Complexity**

We tested several tree depths to understand the relationship between model complexity and validation performance.

The shallow tree (`max_depth=3`) clearly underfits the data.

Increasing the depth substantially improves performance, indicating that the model benefits from learning more complex nonlinear relationships.

However, the improvement becomes progressively smaller:

- Depth 3 → 5: +0.02333
- Depth 5 → 7: +0.00928
- Depth 7 → 10: +0.00175

This suggests that we may be approaching a point of diminishing returns.

Before increasing the tree complexity further, we need to check the model's training performance. A large gap between training and validation ROC AUC would indicate overfitting.

Therefore, our next experiment will compare **training vs. validation ROC AUC** for different tree depths.

In [25]:
# We will compare training and validation ROC AUC
# for several tree depths.
#
# If training AUC becomes much higher than validation AUC,
# the tree may be overfitting.
#
# If both scores remain relatively close, the added complexity
# is still generalizing well.

tree_depths = [3, 5, 7, 10, 15]

for depth in tree_depths:

    # Build a Decision Tree with the current depth.
    tree = Pipeline(
        steps=[
            ("preprocessor", top_preprocessor),
            ("model", DecisionTreeClassifier(
                max_depth=depth,
                random_state=42
            ))
        ]
    )

    # Train the model.
    tree.fit(X_train_top, y_train_top)

    # Get probability predictions for the training set.
    train_predictions = tree.predict_proba(X_train_top)[:, 1]

    # Get probability predictions for the validation set.
    valid_predictions = tree.predict_proba(X_valid_top)[:, 1]

    # Calculate ROC AUC on both datasets.
    train_auc = roc_auc_score(y_train_top, train_predictions)
    valid_auc = roc_auc_score(y_valid_top, valid_predictions)

    # Calculate the train-validation gap.
    auc_gap = train_auc - valid_auc

    print(
        f"max_depth={depth:2d} | "
        f"Train AUC={train_auc:.5f} | "
        f"Validation AUC={valid_auc:.5f} | "
        f"Gap={auc_gap:.5f}"
    )

max_depth= 3 | Train AUC=0.89627 | Validation AUC=0.89531 | Gap=0.00096
max_depth= 5 | Train AUC=0.92001 | Validation AUC=0.91864 | Gap=0.00137
max_depth= 7 | Train AUC=0.92952 | Validation AUC=0.92792 | Gap=0.00160
max_depth=10 | Train AUC=0.93500 | Validation AUC=0.92967 | Gap=0.00533
max_depth=15 | Train AUC=0.95179 | Validation AUC=0.90767 | Gap=0.04412


### **Detecting Overfitting**

We compared training and validation ROC AUC across different tree depths.

The best balance in our experiments is around `max_depth=10`:

- Training ROC AUC: **0.93500**
- Validation ROC AUC: **0.92967**
- Train-validation gap: **0.00533**

After that, with `max_depth=15`, we achieve a higher Train AUC but the Validation AUC lowers significantly, meaning that an increased depth has risked overfitting and causes generalization issues.

We'll save the Validation AUC achieved with our preferred model (`max_depth=10`) for future comparison purposes.

In [26]:
best_tree_auc = 0.92967

## **Random Forest**

We now test a Random Forest while keeping the same features and validation split.

The depth is based on our previous experiment, where a depth of 7 provided a strong balance between validation performance and overfitting.

The purpose of this experiment is to determine whether combining multiple trees improves generalization compared with a single Decision Tree.

In [27]:
# Build a Random Forest using the same three strongest features.
# Using multiple trees should allow the model to capture
# nonlinear patterns more robustly than a single Decision Tree.

rf_model = Pipeline(
    steps=[
        ("preprocessor", top_preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=200,      # Number of trees in the forest
            max_depth=7,           # Start with the depth that generalized well
            random_state=42,
            n_jobs=-1              # Use all available CPU cores
        ))
    ]
)

# Train the Random Forest on the same training split.
rf_model.fit(X_train_top, y_train_top)

# Predict probabilities for the positive class.
rf_valid_predictions = rf_model.predict_proba(
    X_valid_top
)[:, 1]

# Evaluate using the competition metric.
rf_auc = roc_auc_score(
    y_valid_top,
    rf_valid_predictions
)

print(f"Random Forest ROC AUC: {rf_auc:.5f}")
print(f"Decision Tree ROC AUC: {tree_auc:.5f}")
print(f"Improvement: {rf_auc - tree_auc:+.5f}")

Random Forest ROC AUC: 0.92795
Decision Tree ROC AUC: 0.91864
Improvement: +0.00931


### **Feature Selection vs. All Features**

Our initial experiments showed that the three strongest features contain almost all of the predictive signal captured by Logistic Regression.

We'll now do an **ablation-style feature selection experiment**: we compare a restricted feature set against the complete feature set to determine whether the additional variables provide useful predictive information.

In [28]:
# We will now train the Random Forest using all available features.
#
# This experiment answers a very specific question:
# Do the remaining features provide useful nonlinear information
# that the Top-3 model could not capture?
#
# We keep the model configuration and validation split unchanged
# so that the only meaningful difference is the feature set.

all_numeric_features = [
    "age",
    "daily_screen_time_hours",
    "social_media_hours",
    "gaming_hours",
    "work_study_hours",
    "sleep_hours",
    "notifications_per_day",
    "app_opens_per_day",
    "weekend_screen_time"
]

all_categorical_features = [
    "gender",
    "stress_level",
    "academic_work_impact"
]

In [29]:
# Build preprocessing for all features.
# Numerical missing values are replaced with the median.
# Categorical missing values are replaced with the most frequent value,
# then categorical variables are one-hot encoded.
all_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            all_numeric_features
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore"))
                ]
            ),
            all_categorical_features
        )
    ]
)

In [30]:
# Build a Random Forest with the same settings as our Top-3 experiment.
rf_all_model = Pipeline(
    steps=[
        ("preprocessor", all_preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=200,
            max_depth=7,
            random_state=42,
            n_jobs=-1
        ))
    ]
)

# Train using all features.
rf_all_model.fit(X_train, y_train)

# Generate probability scores for the positive class.
rf_all_predictions = rf_all_model.predict_proba(X_valid)[:, 1]

# Evaluate on the validation set.
rf_all_auc = roc_auc_score(y_valid, rf_all_predictions)

print(f"Random Forest - All Features ROC AUC: {rf_all_auc:.5f}")
print(f"Random Forest - Top 3 ROC AUC:       {rf_auc:.5f}")
print(f"Difference:                          {rf_all_auc - rf_auc:+.5f}")

Random Forest - All Features ROC AUC: 0.92649
Random Forest - Top 3 ROC AUC:       0.92795
Difference:                          -0.00146


**Observations**

The Random Forest using the three strongest features achieved a ROC AUC of **0.92795**.

Using all available features reduced the validation ROC AUC to **0.92649**.

This suggests that the additional features do not provide enough useful predictive information to compensate for the extra noise or complexity they introduce.

An important lesson is that adding more features does not necessarily improve a machine learning model.

For this dataset, the three main screen-usage variables currently provide the strongest feature set:

- `daily_screen_time_hours`
- `weekend_screen_time`
- `social_media_hours`

We will therefore keep the Top-3 feature set for the next experiments and avoid introducing additional variables unless an experiment provides evidence that they improve validation performance.

### **Random Forest Hyperparameter Optimization**

Our initial Random Forest did not outperform the best Decision Tree, but this does not mean that Random Forest is inherently worse. Its performance depends on several hyperparameters that control the complexity and diversity of the trees.

We use `RandomizedSearchCV` to explore combinations of:

- `n_estimators`: number of trees
- `max_depth`: maximum tree depth
- `min_samples_leaf`: minimum number of samples in a leaf
- `max_features`: number of features considered at each split

The search uses 3-fold cross-validation on the training data.

The validation set remains completely untouched during hyperparameter optimization. This allows us to use it as an independent estimate of how well the selected configuration generalizes.

Our current benchmark is the Decision Tree with `max_depth=10`, which achieved a validation ROC AUC of **0.92967**.

The goal of this experiment is to determine whether an optimized Random Forest can surpass that benchmark.

In [33]:
# We will search for better Random Forest hyperparameters.
#
# We focus on the parameters that control:
# - forest size (`n_estimators`)
# - tree complexity (`max_depth`)
# - minimum leaf size (`min_samples_leaf`)
# - number of features considered at each split (`max_features`)
#
# RandomizedSearchCV samples combinations instead of exhaustively
# testing every possible combination, making the search more efficient.

#rf_param_grid = {
    #"model__n_estimators": [200, 400, 600],
    #"model__max_depth": [5, 7, 9, 11, None],
    #"model__min_samples_leaf": [1, 2, 5, 10],
    #"model__max_features": ["sqrt", "log2", 0.8, 1.0]
#}

# RandomizedSearchCV will evaluate several randomly selected
# combinations using stratified cross-validation.
#
# We use ROC AUC because it is the competition metric.
#rf_random_search = RandomizedSearchCV(
    #estimator=rf_model,
    #param_distributions=rf_param_grid,
    #n_iter=12,
    #scoring="roc_auc",
    #cv=3,
    #random_state=42,
    #n_jobs=-1,
    #verbose=1
#)

# Run the hyperparameter search on the training split only.
#rf_random_search.fit(X_train_top, y_train_top)

# Display the best configuration found during cross-validation.
#print("Best parameters:")
#print(rf_random_search.best_params_)

#print(f"\nBest CV ROC AUC: {rf_random_search.best_score_:.5f}")


In [34]:
# Save the best Random Forest pipeline to disk.
# This stores both the preprocessing steps and the trained model,
# so we do not need to run the 20-minute hyperparameter search again.

#joblib.dump(
    #rf_random_search.best_estimator_,
    #"best_random_forest.joblib"
#)

#print("Best Random Forest saved successfully.")

In [35]:
# Evaluate the best model on our untouched validation set.
#best_rf = rf_random_search.best_estimator_

best_rf_predictions = best_rf.predict_proba(
    X_valid_top
)[:, 1]

best_rf_valid_auc = roc_auc_score(
    y_valid_top,
    best_rf_predictions
)

print(f"Validation ROC AUC: {best_rf_valid_auc:.5f}")
print(f"Current best ROC AUC: {best_tree_auc:.5f}")
print(f"Improvement vs. current best: {best_rf_valid_auc - best_tree_auc:+.5f}")

Validation ROC AUC: 0.93343
Current best ROC AUC: 0.92967
Improvement vs. current best: +0.00376


RandomizedSearchCV found a stronger Random Forest configuration:

- `n_estimators = 400`
- `max_depth = 11`
- `min_samples_leaf = 2`
- `max_features = 0.8`

The best cross-validation ROC AUC was **0.93418**, while the performance on our untouched validation set was **0.93343**. This represents an improvement over our best Decision Tree.

The small difference between cross-validation and validation performance suggests that the model is generalizing consistently.

The result confirms that combining multiple nonlinear trees and tuning their complexity provides additional predictive power over a single Decision Tree.

Our current benchmark is therefore **0.93343 ROC AUC**.

### **Random Forest Feature Importance**

After selecting and tuning our Random Forest, we examine which features the model relies on most heavily.

Feature importance helps us compare the model's learned behavior with the hypotheses developed during EDA.

Our main expectation is that:

- `daily_screen_time_hours`
- `weekend_screen_time`
- `social_media_hours`

will account for most of the model's predictive structure.

However, feature importance should not be interpreted as causal importance. It tells us how useful a feature was for the model's decisions, not whether the feature causes smartphone addiction.

We will use this analysis primarily as a consistency check between our exploratory analysis and the final model's behavior.

In [36]:
# Extract the trained Random Forest from the pipeline.
# This gives us direct access to the fitted tree ensemble.
rf_estimator = best_rf.named_steps["model"]

# Get the feature names produced by the preprocessing step.
# In our case, the model uses only the three numerical Top-3 features,
# so these names correspond directly to those original columns.
feature_names = best_rf.named_steps["preprocessor"].get_feature_names_out()

# Create a table with each feature and its average importance
# across the trees in the Random Forest.
feature_importance = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": rf_estimator.feature_importances_
    })
    .sort_values("importance", ascending=False)
)

# Display the feature importances.
feature_importance

,feature,importance
0,daily_screen_time_hours,0.510218
1,weekend_screen_time,0.269811
2,social_media_hours,0.219971


**Observations**

The optimized Random Forest assigns all of its predictive importance to our three selected features:

- `daily_screen_time_hours`: **51.02%**
- `weekend_screen_time`: **26.98%**
- `social_media_hours`: **22.00%**

This strongly supports our earlier feature-selection analysis.

The model confirms that the main predictive signal is concentrated in different measures of screen usage. The remaining features did not improve the Random Forest when they were included.

This consistency between EDA, model performance, and feature importance increases our confidence that the Top-3 feature set is appropriate for this problem.

Rather than adding weak features back into the model, the next step is to investigate whether we can extract additional information by **engineering new features from these three variables**.



## **Feature Engineering**

Our current model relies almost entirely on three raw screen-usage variables.

We now create two derived features to capture relationships between those variables:

**`weekend_vs_daily_ratio`**

This represents how large weekend screen usage is relative to daily screen usage.

**`social_media_share`**

This represents the proportion of daily screen time associated with social media.

These engineered features provide information about the **composition and relative usage patterns**, rather than only the absolute number of hours.

The purpose of this experiment is to determine whether these relationships contain additional predictive information that the original features do not provide directly.

We will keep the Random Forest hyperparameters unchanged so that any change in ROC AUC can be attributed primarily to the new features.

In [37]:
# Create copies of the Top-3 datasets so we can safely add
# engineered features without modifying the original data.

X_train_fe = X_train_top.copy()
X_valid_fe = X_valid_top.copy()

# Measure how much weekend screen time exceeds daily screen time.
# A larger ratio means relatively heavier phone usage on weekends.
X_train_fe["weekend_vs_daily_ratio"] = (
    X_train_fe["weekend_screen_time"] /
    X_train_fe["daily_screen_time_hours"]
)

X_valid_fe["weekend_vs_daily_ratio"] = (
    X_valid_fe["weekend_screen_time"] /
    X_valid_fe["daily_screen_time_hours"]
)

# Measure the share of total daily screen time spent on social media.
# This gives the model information about the composition of screen time
X_train_fe["social_media_share"] = (
    X_train_fe["social_media_hours"] /
    X_train_fe["daily_screen_time_hours"]
)

X_valid_fe["social_media_share"] = (
    X_valid_fe["social_media_hours"] /
    X_valid_fe["daily_screen_time_hours"]
)

# Display a few rows so we can inspect the new features.
X_train_fe.head()

,daily_screen_time_hours,weekend_screen_time,social_media_hours,weekend_vs_daily_ratio,social_media_share
563697,6.74,10.14,NaN,1.504451,NaN
673485,NaN,5.67,1.37,NaN,NaN
440270,7.79,7.96,2.42,1.021823,0.310655
378032,3.46,5.46,1.17,1.578035,0.338150
446226,NaN,10.58,2.19,NaN,NaN


### **Feature Engineering Evaluation**

We created two additional features that describe relative phone-usage behavior:

- `weekend_vs_daily_ratio`
- `social_media_share`

We now evaluate whether these engineered features provide predictive information beyond the original three features.

The Random Forest hyperparameters remain unchanged from our optimized model, allowing us to isolate the effect of feature engineering.

Our current benchmark is **0.93343 ROC AUC**.

If the engineered-feature model improves validation ROC AUC, we will keep the new features for subsequent experiments.

If performance remains unchanged or decreases, we will discard them and return to the simpler Top-3 feature set.

In [38]:
# We will train the same optimized Random Forest,
# but now using our two engineered features as well.
#
# The goal is to test whether ratios describing usage patterns
# contain additional predictive information beyond the three
# original features.

rf_feature_engineered = Pipeline(
    steps=[
        ("preprocessor", Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median"))
            ]
        )),
        ("model", RandomForestClassifier(
            n_estimators=400,
            max_depth=11,
            min_samples_leaf=2,
            max_features=0.8,
            random_state=42,
            n_jobs=-1
        ))
    ]
)

# Train the model using the engineered features.
rf_feature_engineered.fit(
    X_train_fe,
    y_train_top
)

# Generate probability predictions for the validation set.
fe_predictions = rf_feature_engineered.predict_proba(
    X_valid_fe
)[:, 1]

# Evaluate using the competition metric.
fe_auc = roc_auc_score(
    y_valid_top,
    fe_predictions
)

print(f"Feature-Engineered RF ROC AUC: {fe_auc:.5f}")
print(f"Current Best RF ROC AUC:       {best_rf_valid_auc:.5f}")
print(f"Difference:                    {fe_auc - best_rf_valid_auc:+.5f}")

Feature-Engineered RF ROC AUC: 0.93312
Current Best RF ROC AUC:       0.93343
Difference:                    -0.00030


**Observations**

The feature-engineered Random Forest achieved a ROC AUC of **0.93312**, compared with **0.93343** for the original Top-3 feature set.

The difference is **-0.00030 ROC AUC**, meaning the engineered features did not provide additional predictive value in this experiment.

Therefore, we will continue with the original three features:

- `daily_screen_time_hours`
- `weekend_screen_time`
- `social_media_hours`

## **Gradient Boosting: XGBoost**

Random Forest improved substantially over a single Decision Tree. We now test Gradient Boosting with XGBoost.

We will initially keep the same three selected features and use a reasonable set of hyperparameters rather than performing another expensive search immediately.

Our benchmark remains:

**Random Forest: 0.93343 ROC AUC**

In [41]:
# Build an initial XGBoost model using the same Top-3 features.
# We keep the feature set unchanged so that the comparison with
# Random Forest isolates the effect of the learning algorithm.

xgb_model = Pipeline(
    steps=[
        ("preprocessor", top_preprocessor),
        ("model", XGBClassifier(
            n_estimators=500,          # Number of boosting rounds
            max_depth=6,               # Moderate tree complexity
            learning_rate=0.05,        # Small steps can improve generalization
            subsample=0.8,             # Use 80% of rows for each boosting round
            colsample_bytree=1.0,      # Use all available features at each tree
            objective="binary:logistic",
            eval_metric="auc",
            random_state=42,
            n_jobs=-1
        ))
    ]
)

# Train XGBoost using the same training split as our previous models.
xgb_model.fit(
    X_train_top,
    y_train_top
)

# Generate probability scores for the positive class.
xgb_predictions = xgb_model.predict_proba(
    X_valid_top
)[:, 1]

# Evaluate using the competition metric.
xgb_auc = roc_auc_score(
    y_valid_top,
    xgb_predictions
)

print(f"XGBoost ROC AUC:        {xgb_auc:.5f}")
print(f"Random Forest ROC AUC:  {best_rf_valid_auc:.5f}")
print(f"Difference:             {xgb_auc - best_rf_valid_auc:+.5f}")

XGBoost ROC AUC:        0.93542
Random Forest ROC AUC:  0.93343
Difference:             +0.00199


**Observations**

XGBoost achieved a validation ROC AUC of **0.93542**, outperforming our optimized Random Forest at **0.93343**.

This represents an improvement of **0.00199 ROC AUC**.

Although the gain is relatively modest, it is consistent and meaningful enough to justify further investigation of XGBoost.

The result suggests that sequential boosting is able to capture additional predictive patterns that the Random Forest does not fully capture.

Our new benchmark is therefore:

**XGBoost: 0.93542 ROC AUC**

We will now focus on optimizing XGBoost rather than continuing to tune the Random Forest.

### **XGBoost Hyperparameter Exploration**

We now investigate two important XGBoost hyperparameters:

- `max_depth`: controls the complexity of individual trees.
- `learning_rate`: controls how strongly each new tree contributes to the ensemble.

A higher `max_depth` allows the model to learn more complex interactions, but excessive depth can increase overfitting.

A lower `learning_rate` makes learning slower and often requires more boosting rounds, but it can improve generalization.

We test a small number of configurations while keeping the other parameters fixed.

This is an exploratory step rather than a full hyperparameter search. Its purpose is to identify promising regions of the parameter space before performing more expensive optimization.

In [42]:
# We will test a small set of XGBoost configurations.
#
# The goal is to understand how tree complexity (`max_depth`)
# and learning speed (`learning_rate`) affect generalization.
#
# We keep the number of boosting rounds and sampling parameters
# fixed so that we can focus on these two hyperparameters.

xgb_configs = [
    {"max_depth": 4, "learning_rate": 0.05},
    {"max_depth": 6, "learning_rate": 0.05},
    {"max_depth": 8, "learning_rate": 0.05},
    {"max_depth": 6, "learning_rate": 0.02},
    {"max_depth": 6, "learning_rate": 0.10},
]

for config in xgb_configs:

    # Build an XGBoost model using the current configuration.
    model = Pipeline(
        steps=[
            ("preprocessor", top_preprocessor),
            ("model", XGBClassifier(
                n_estimators=500,
                max_depth=config["max_depth"],
                learning_rate=config["learning_rate"],
                subsample=0.8,
                colsample_bytree=1.0,
                objective="binary:logistic",
                eval_metric="auc",
                random_state=42,
                n_jobs=-1
            ))
        ]
    )

    # Train the model on the same training split.
    model.fit(X_train_top, y_train_top)

    # Generate positive-class probabilities.
    predictions = model.predict_proba(X_valid_top)[:, 1]

    # Evaluate using ROC AUC.
    auc = roc_auc_score(y_valid_top, predictions)

    print(
        f"max_depth={config['max_depth']:2d} | "
        f"learning_rate={config['learning_rate']:.2f} | "
        f"Validation AUC={auc:.5f}"
    )

max_depth= 4 | learning_rate=0.05 | Validation AUC=0.93418
max_depth= 6 | learning_rate=0.05 | Validation AUC=0.93542
max_depth= 8 | learning_rate=0.05 | Validation AUC=0.93563
max_depth= 6 | learning_rate=0.02 | Validation AUC=0.93363
max_depth= 6 | learning_rate=0.10 | Validation AUC=0.93572


## **Exploring CatBoost**

XGBoost performance remained relatively stable across the tested
`max_depth` and `learning_rate` configurations.

The best result was:

- `max_depth=6`
- `learning_rate=0.10`
- ROC AUC = **0.93572**

The differences between configurations are small, suggesting that further
manual tuning of these two parameters is unlikely to produce a major gain.

We therefore move to a different boosting algorithm: **CatBoost**.

CatBoost uses a different boosting strategy and is particularly well suited
to structured/tabular data and categorical variables.

We will first compare it using our established Top-3 feature set to isolate
the effect of the algorithm itself.

In [45]:
# We start with CatBoost using the same three strongest features.

cat_model = CatBoostClassifier(
    iterations=500,          # Number of boosting iterations
    depth=6,                 # Moderate tree complexity
    learning_rate=0.05,      # Conservative learning rate
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=False,
    thread_count=-1
)

# We train on the already selected features.
cat_model.fit(
    X_train_top,
    y_train_top
)

# Generate probability scores for the positive class.
cat_predictions = cat_model.predict_proba(
    X_valid_top
)[:, 1]

# Evaluate using the competition metric.
cat_auc = roc_auc_score(
    y_valid_top,
    cat_predictions
)

print(f"CatBoost ROC AUC:   {cat_auc:.5f}")
print(f"XGBoost ROC AUC:    {0.93572:.5f}")
print(f"Difference:         {cat_auc - 0.93572:+.5f}")

CatBoost ROC AUC:   0.93403
XGBoost ROC AUC:    0.93572
Difference:         -0.00169


### **XGBoost with All Features**

CatBoost did not outperform our current XGBoost benchmark.

Before moving to another modeling family, we test whether XGBoost can extract
useful nonlinear interactions from the features that were excluded from our
Top-3 model.

The previous Random Forest experiment showed that adding all features slightly
reduced performance. However, different algorithms can exploit additional
features in different ways.

We therefore test XGBoost using all available features while keeping the same
model configuration as our best current XGBoost experiment.

Our current benchmark is **0.93572 ROC AUC**.

In [46]:
# We will test XGBoost using all available features.
#
# This experiment isolates the effect of the feature set:
# the model architecture remains the same, while we add the
# features that were previously excluded.

X_train_all = X_train.copy()
X_valid_all = X_valid.copy()

# Build preprocessing for all numerical and categorical features.
# Numerical missing values are replaced with the median.
# Categorical values are imputed and one-hot encoded.
xgb_all_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            all_numeric_features
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore"))
                ]
            ),
            all_categorical_features
        )
    ]
)

# Build XGBoost using the same configuration as our current
# strongest XGBoost experiment.
xgb_all_model = Pipeline(
    steps=[
        ("preprocessor", xgb_all_preprocessor),
        ("model", XGBClassifier(
            n_estimators=500,
            max_depth=6,
            learning_rate=0.10,
            subsample=0.8,
            colsample_bytree=1.0,
            objective="binary:logistic",
            eval_metric="auc",
            random_state=42,
            n_jobs=-1
        ))
    ]
)

# Train using all available features.
xgb_all_model.fit(X_train_all, y_train)

# Generate positive-class probabilities.
xgb_all_predictions = xgb_all_model.predict_proba(
    X_valid_all
)[:, 1]

# Evaluate using ROC AUC.
xgb_all_auc = roc_auc_score(
    y_valid,
    xgb_all_predictions
)

print(f"XGBoost - All Features ROC AUC: {xgb_all_auc:.5f}")
print(f"XGBoost - Top 3 ROC AUC:       {0.93572:.5f}")
print(f"Difference:                    {xgb_all_auc - 0.93572:+.5f}")

XGBoost - All Features ROC AUC: 0.96190
XGBoost - Top 3 ROC AUC:       0.93572
Difference:                    +0.02618


**Observations**

The all-feature XGBoost model achieved a ROC AUC of **0.96190**, compared with **0.93572** using only the Top-3 features.

This represents a substantial improvement of **0.02618 ROC AUC**.

This result changes our previous conclusion about feature selection.

The weaker features identified during our univariate analysis were not necessarily useless. Instead, they appear to contain information that becomes valuable when XGBoost can model nonlinear relationships and interactions between variables.

This highlights an important limitation of evaluating features independently:

> A feature with weak individual correlation with the target can still provide substantial predictive value through interactions with other features.

At this stage, XGBoost using all features becomes our new benchmark:

**ROC AUC = 0.96190**

### **Checking XGBoost Generalization**

Our all-feature XGBoost model achieved a validation ROC AUC of **0.96190**.

Before further tuning, we check the difference between training and validation performance, to see how well our model is generalizing.

This diagnostic will guide our next optimization step:

- If the gap is small, we can focus on increasing model capacity or improving the boosting configuration.
- If the gap is large, we should focus on regularization and controlling model complexity.

We use this analysis before performing further tuning so that optimization decisions are based on observed model behavior rather than trial and error.

In [48]:
# We compare training and validation ROC AUC for our strongest XGBoost model.
#
# This tells us whether the model is generalizing well or whether
# its high validation score is accompanied by a large overfitting gap.

xgb_train_predictions = xgb_all_model.predict_proba(
    X_train_all
)[:, 1]

xgb_valid_predictions = xgb_all_model.predict_proba(
    X_valid_all
)[:, 1]

# Calculate ROC AUC on both datasets.
xgb_train_auc = roc_auc_score(
    y_train,
    xgb_train_predictions
)

xgb_valid_auc = roc_auc_score(
    y_valid,
    xgb_valid_predictions
)

# Measure the difference between training and validation performance.
xgb_gap = xgb_train_auc - xgb_valid_auc

print(f"XGBoost Train AUC:      {xgb_train_auc:.5f}")
print(f"XGBoost Validation AUC: {xgb_valid_auc:.5f}")
print(f"Train-Validation Gap:   {xgb_gap:.5f}")

XGBoost Train AUC:      0.96935
XGBoost Validation AUC: 0.96190
Train-Validation Gap:   0.00745


### **XGBoost Regularization**

Our current XGBoost model achieves a validation ROC AUC of **0.96190**, with a train-validation gap of **0.00745**.

This indicates some degree of overfitting.

We therefore test several regularization configurations using:

- `min_child_weight`
- `subsample`
- `reg_lambda`

These parameters can make the model less likely to learn highly specific patterns from the training data.

We compare both training and validation ROC AUC because the goal is not simply to increase training performance. We want to improve validation performance while maintaining a reasonable generalization gap.

The current benchmark remains **0.96190 ROC AUC**.

In [50]:
# We will test a few regularized XGBoost configurations.
#
# Our current model has a validation AUC of 0.96190 and a
# train-validation gap of 0.00745.
#
# The goal is to reduce overfitting while preserving or improving
# validation performance.
#
# We keep the main boosting configuration fixed and only modify
# parameters that directly affect regularization.

regularization_configs = [
    {
        "min_child_weight": 1,
        "subsample": 0.8,
        "reg_lambda": 1
    },
    {
        "min_child_weight": 5,
        "subsample": 0.8,
        "reg_lambda": 5
    },
    {
        "min_child_weight": 10,
        "subsample": 0.7,
        "reg_lambda": 10
    }
]

for config in regularization_configs:

    # Build XGBoost using the current regularization configuration.
    model = Pipeline(
        steps=[
            ("preprocessor", xgb_all_preprocessor),
            ("model", XGBClassifier(
                n_estimators=500,
                max_depth=6,
                learning_rate=0.10,

                # Regularization parameters being tested.
                min_child_weight=config["min_child_weight"],
                subsample=config["subsample"],
                reg_lambda=config["reg_lambda"],

                # Keep the remaining parameters unchanged.
                colsample_bytree=1.0,
                objective="binary:logistic",
                eval_metric="auc",
                random_state=42,
                n_jobs=-1
            ))
        ]
    )

    # Train using the same training split as all previous experiments.
    model.fit(X_train_all, y_train)

    # Generate predictions for both datasets.
    train_predictions = model.predict_proba(X_train_all)[:, 1]
    valid_predictions = model.predict_proba(X_valid_all)[:, 1]

    # Calculate ROC AUC for training and validation.
    train_auc = roc_auc_score(y_train, train_predictions)
    valid_auc = roc_auc_score(y_valid, valid_predictions)

    # Calculate the generalization gap.
    gap = train_auc - valid_auc

    print(
        f"min_child_weight={config['min_child_weight']:2d} | "
        f"subsample={config['subsample']:.1f} | "
        f"reg_lambda={config['reg_lambda']:2d} | "
        f"Train AUC={train_auc:.5f} | "
        f"Validation AUC={valid_auc:.5f} | "
        f"Gap={gap:.5f}"
    )

min_child_weight= 1 | subsample=0.8 | reg_lambda= 1 | Train AUC=0.96935 | Validation AUC=0.96190 | Gap=0.00745
min_child_weight= 5 | subsample=0.8 | reg_lambda= 5 | Train AUC=0.96787 | Validation AUC=0.96173 | Gap=0.00614
min_child_weight=10 | subsample=0.7 | reg_lambda=10 | Train AUC=0.96701 | Validation AUC=0.96166 | Gap=0.00535


**Observations**

The original XGBoost configuration achieved the best validation ROC AUC:

**0.96190**

Increasing regularization reduced the train-validation gap:

- Original: **0.00745**
- Moderate regularization: **0.00614**
- Strong regularization: **0.00535**

However, both regularized models also produced slightly lower validation ROC AUC values.

Therefore, although stronger regularization improved generalization in terms of the train-validation gap, it did not improve our competition metric.

We will keep the original XGBoost configuration as our current best model:

**Validation ROC AUC = 0.96190**

At this point, further manual tuning is unlikely to provide enough benefit relative to the remaining time. We will move toward final model selection and submission.

In [51]:
# We already have validation probabilities from our two strongest models:
# XGBoost and Random Forest.
#
# Instead of training another model, we test whether combining their
# predictions can improve the ranking of positive vs. negative cases.
#
# Because XGBoost is substantially stronger, we give it most of the weight.

blend_weights = [0.75, 0.90, 1.00]

for xgb_weight in blend_weights:

    # The Random Forest receives the remaining weight.
    rf_weight = 1 - xgb_weight

    # Combine the probability predictions from both models.
    blended_predictions = (
        xgb_weight * xgb_valid_predictions
        + rf_weight * best_rf_predictions
    )

    # Evaluate the blended predictions using ROC AUC.
    blend_auc = roc_auc_score(
        y_valid,
        blended_predictions
    )

    print(
        f"XGBoost weight={xgb_weight:.2f} | "
        f"Random Forest weight={rf_weight:.2f} | "
        f"ROC AUC={blend_auc:.5f}"
    )

XGBoost weight=0.75 | Random Forest weight=0.25 | ROC AUC=0.95975
XGBoost weight=0.90 | Random Forest weight=0.10 | ROC AUC=0.96130
XGBoost weight=1.00 | Random Forest weight=0.00 | ROC AUC=0.96190


## **Final XGBoost Model**

After comparing several model families, feature sets, and XGBoost configurations, the all-feature XGBoost model achieved the best validation ROC AUC of **0.96190**.

Additional regularization reduced the train-validation gap but slightly decreased validation ROC AUC, so the original configuration was retained.

Blending XGBoost predictions with the optimized Random Forest also failed to improve ROC AUC.

Therefore, XGBoost was selected as the final model.

The final model is now trained using the complete training dataset so that it can learn from all available labeled observations before generating predictions for the competition test set.

In [52]:
# Train the final XGBoost model using the complete training dataset.
#
# We no longer need a validation split because model selection
# has already been completed using X_valid_all and y_valid.

final_xgb = Pipeline(
    steps=[
        ("preprocessor", xgb_all_preprocessor),
        ("model", XGBClassifier(
            n_estimators=500,
            max_depth=6,
            learning_rate=0.10,

            # Best regularization configuration found.
            min_child_weight=1,
            subsample=0.8,
            reg_lambda=1,

            colsample_bytree=1.0,
            objective="binary:logistic",
            eval_metric="auc",
            random_state=42,
            n_jobs=-1
        ))
    ]
)

# Separate features and target from the complete training dataset.
X_full = train.drop(columns="addicted_label")
y_full = train["addicted_label"]

# Train the final model on all available training data.
final_xgb.fit(X_full, y_full)

print("Final XGBoost model trained successfully.")

Final XGBoost model trained successfully.


## **Final Predictions and Submission**

The final XGBoost model was used to generate probability predictions for the competition test set.

Because the competition metric is ROC AUC, we submitted the predicted probability of `addicted_label = 1` rather than converting predictions into binary 0/1 labels.

The submission contains the required `id` column and the predicted `addicted_label` probabilities.

The file was saved as `submission.csv` and is ready to upload to Kaggle.

In [53]:
# Generate probability predictions for the competition test set.
#
# We use predict_proba because the competition evaluates ROC AUC,
# which requires continuous prediction scores rather than hard labels.

X_test = test.copy()

test_predictions = final_xgb.predict_proba(X_test)[:, 1]

# Create the submission dataframe using the required columns.
submission = pd.DataFrame({
    "id": test["id"],
    "addicted_label": test_predictions
})

# Save the submission file.
submission.to_csv("submission.csv", index=False)

# Check the resulting submission.
print("Submission shape:", submission.shape)
print("\nFirst 5 predictions:")
print(submission.head())

print("\nPrediction range:")
print(
    f"Min: {submission['addicted_label'].min():.5f} | "
    f"Max: {submission['addicted_label'].max():.5f}"
)

Submission shape: (296302, 2)

First 5 predictions:
       id  addicted_label
0  691369        0.999325
1  691370        0.947213
2  691371        0.960887
3  691372        0.987810
4  691373        0.998503

Prediction range:
Min: 0.00009 | Max: 1.00000
